In [1]:
import sys
import numpy as np

sys.path.append("../../")
from Rain import Rain

sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-06-30 22:28:53.883288: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-06-30 22:28:54.598372: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import sys
sys.path.append('../../')
from clean_all import clean
clean()
sys.path.pop()

'../../'

In [3]:
config = {
    "lib": "tensorflow",
    "partitions": 1,
    "iterations": 3,
    "lr": 0.001,
    "epochs": 3,
    "batch_size": 128,
    "loss": tf.keras.losses.CategoricalCrossentropy(),
    "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),
}

In [4]:
def get_train_data():
    return np.load("../../data/MNIST/train_data.npy"), np.load(
        "../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../data/MNIST/test_data.npy"), np.load(
        "../../data/MNIST/test_labels.npy"
    )


def partition_train_data(X_train, y_train, partitions):
    num_samples = X_train.shape[0]

    # Create an array of indices from 0 to num_samples - 1
    indices = np.arange(num_samples)

    # Shuffle the indices
    np.random.shuffle(indices)

    # Use the shuffled indices to shuffle the datasets
    X_train = X_train[indices]
    y_train = y_train[indices]

    X_train_partitions = []
    y_train_partitions = []

    partition_size = int(len(X_train) / partitions)

    for i in range(partitions):
        if i == partitions - 1:
            X_train_partitions.append(X_train[i * partition_size :])
            y_train_partitions.append(y_train[i * partition_size :])
        else:
            X_train_partitions.append(
                X_train[i * partition_size : (i + 1) * partition_size]
            )
            y_train_partitions.append(
                y_train[i * partition_size : (i + 1) * partition_size]
            )

    return X_train_partitions, y_train_partitions

In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()
X_train, y_train = partition_train_data(X_train, y_train, config["partitions"])

In [7]:
for i in range(len(X_train)):
    np.save(f"../../data/X_train_{i + 1}.npy", X_train[i])
    np.save(f"../../data/y_train_{i + 1}.npy", y_train[i])

In [ ]:
model = create_model()
rain = Rain(config, model, X_train, y_train)

Rain is initialized
Provisioner created successfully
divider is running


In [ ]:
# rain.setup_vms()

In [ ]:
# rain.delete_vms()

In [ ]:
model = rain.train_centralized_sync()

divider received: Success receiving the number of workers
divider is sending data to the coordinator
 divider received: File received successfully
 divider received: File received successfully
 divider received: File received successfully
Starting iteration 1/3
sending file:  ../../data/1.pkl
divider is sending information file to the coordinator
 divider received: File received successfully
divider is beginning the iteration
filepath is: ../../Divider/divider/data/1_trained.pkl
 divider received: one loop is done
gradients:  [[array([[-0., -0., -0., ..., -0., -0., -0.],
       [-0., -0., -0., ..., -0., -0., -0.],
       [-0., -0., -0., ..., -0., -0., -0.],
       ...,
       [-0., -0., -0., ..., -0., -0., -0.],
       [-0., -0., -0., ..., -0., -0., -0.],
       [-0., -0., -0., ..., -0., -0., -0.]], dtype=float32), array([ 3.13020267e+01, -1.85797558e+01,  5.23244972e+01,  2.43874569e+01,
        4.43519096e+01, -2.51601143e+01, -3.53800163e+01,  3.39808807e+01,
        1.43776865e+01,

In [ ]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 2ms/step - loss: 0.5817 - accuracy: 0.9697

Test accuracy: 97.0%
